In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

pd.set_option('display.max_columns', 50)
pd.set_option('display.max_rows', 100)

In [2]:
# File paths
BV  = 'data/tmpz1yo1ola.csv'
PW  = 'data/tmpizrf688w.csv'
SAM = 'data/live_street_address_management_sam_addresses.csv'
PA  = 'data/fy2026-property-assessment-data_12_23_2025.csv'

# Load datasets
bv  = pd.read_csv(BV,  dtype={'sam_id': str})
pw  = pd.read_csv(PW,  dtype={'sam_id': str})
sam = pd.read_csv(SAM, dtype={'SAM_ADDRESS_ID': str, 'PARCEL': str}, low_memory=False)
pa  = pd.read_csv(PA,  dtype={'PID': str}, low_memory=False)
pa.columns = pa.columns.str.strip()

In [3]:
print(f'Building Violations before cleaning: {len(bv):,} rows')

# Drop rows with sam_id missing
bv_clean = bv.dropna(subset=['sam_id'])

# Drop dups
bv_clean = bv_clean.drop_duplicates()

# Strip whitespace
bv_clean['sam_id'] = bv_clean['sam_id'].str.strip()

# Parse violation dates
date_col = [c for c in bv_clean.columns if 'date' in c.lower() or 'dttm' in c.lower()]
for col in date_col:
    bv_clean[col] = pd.to_datetime(bv_clean[col], errors='coerce')

# Standardize: strip whitespace
str_cols = bv_clean.select_dtypes(include='str').columns
for col in str_cols:
    bv_clean[col] = bv_clean[col].str.strip()

bv_clean['source'] = 'building_violations'

print(f'\nBuilding Violations after cleaning: {len(bv_clean):,} rows')
bv_clean.head(3)

Building Violations before cleaning: 17,238 rows

Building Violations after cleaning: 17,129 rows


,case_no,ap_case_defn_key,status_dttm,status,code,value,description,violation_stno,violation_sthigh,violation_street,violation_suffix,violation_city,violation_state,violation_zip,ward,contact_addr1,contact_addr2,contact_city,contact_state,contact_zip,sam_id,latitude,longitude,location,source
0,V91983,1013,NaT,Closed,121.2,NaN,Unsafe and Dangerous,302,NaN,Sumner,ST,East Boston,MA,02128,01,302 Sumner St,NaN,East Boston,MA,02128,132380,42.367679,-71.036580,"(42.36767899977675, -71.03658000178699)",building_violations
1,V903743,1013,2026-04-21 10:19:12,Open,102.8,NaN,Maintenance,125,NaN,B,ST,South Boston,MA,02127,06,125 B ST,C/O LAWRENCE ASSOCIATES LLC,SOUTH BOSTON,MA,02127,7362,42.341270,-71.053080,"(42.341270000257374, -71.05308000068847)",building_violations
2,V903738,1013,2026-04-21 10:07:25,Open,102.8,NaN,Maintenance,105,NaN,Third,ST,South Boston,MA,02127,06,836 EAST THIRD ST,NaN,SOUTH BOSTON,MA,02127,135205,42.341123,-71.052757,"(42.34112297848231, -71.05275722920858)",building_violations


In [4]:
print(f'Public Works Violations before cleaning: {len(pw):,} rows')

# Drop rows with sam_id missing
pw_clean = pw.dropna(subset=['sam_id'])

# Drop dups
pw_clean = pw_clean.drop_duplicates()

# Strip whitespace
pw_clean['sam_id'] = pw_clean['sam_id'].str.strip()

# Parse violation dates
date_col = [c for c in pw_clean.columns if 'date' in c.lower() or 'dttm' in c.lower()]
for col in date_col:
    pw_clean[col] = pd.to_datetime(pw_clean[col], errors='coerce')

# Standardize: strip whitespace
str_cols = pw_clean.select_dtypes(include='str').columns
for col in str_cols:
    pw_clean[col] = pw_clean[col].str.strip()

pw_clean['source'] = 'public_works'

print(f'\nPublic Works Violations after cleaning: {len(pw_clean):,} rows')
pw_clean.head(3)

Public Works Violations before cleaning: 888,910 rows

Public Works Violations after cleaning: 888,549 rows


,case_no,ticket_no,status_dttm,status,code,value,description,violation_stno,violation_sthigh,violation_street,violation_suffix,violation_city,violation_state,violation_zip,ward,contact_addr1,contact_addr2,contact_city,contact_state,contact_zip,sam_id,latitude,longitude,source
0,CE904845,FEAVLE16,2026-04-22 00:14:00,Open,3,100,Overfilling of barrel/dumpster,273,283,Summer,ST,Boston,MA,2210.0,6,PO BOX 4900 - DEPT 207,C/O RYAN LLC TAX COMPLIANCE,SCOTTSDALE,AZ,85261,460020,42.349883,-71.050209,public_works
1,CE904845,FEAVLE16,2026-04-22 00:14:00,Open,2,50,Improper storage trash: com,273,283,Summer,ST,Boston,MA,2210.0,6,PO BOX 4900 - DEPT 207,C/O RYAN LLC TAX COMPLIANCE,SCOTTSDALE,AZ,85261,460020,42.349883,-71.050209,public_works
2,CE904844,L3VYRM10,2026-04-21 23:45:00,Open,1,25,Improper storage trash: res,35,37,Garden,ST,Boston,MA,2114.0,5,96 Wren Terrace,NaN,Quincy,MA,02169,62482,42.360170,-71.067030,public_works


In [5]:
# Combine BV and PW on their shared columns (BV-only: ap_case_defn_key, location; PW-only: ticket_no)
shared_cols = [c for c in bv_clean.columns if c in pw_clean.columns]
violations = pd.concat([bv_clean[shared_cols], pw_clean[shared_cols]], ignore_index=True)

print(f'Combined violations: {len(violations):,} rows')
print(f'  Building Violations: {len(bv_clean):,}')
print(f'  Public Works Violations: {len(pw_clean):,}')
violations['source'].value_counts()

Combined violations: 905,678 rows
  Building Violations: 17,129
  Public Works Violations: 888,549


source
public_works           888549
building_violations     17129
Name: count, dtype: int64

In [6]:
print(f'SAM before cleaning: {len(sam):,} rows')

# Drop rows missing either join key
sam_clean = sam.dropna(subset=['SAM_ADDRESS_ID', 'PARCEL'])

# Drop dups
sam_clean = sam_clean.drop_duplicates(subset=['SAM_ADDRESS_ID'])

# Convert both join keys to string so they match
sam_clean['SAM_ADDRESS_ID'] = sam_clean['SAM_ADDRESS_ID'].astype(str).str.strip()
sam_clean['PARCEL'] = sam_clean['PARCEL'].astype(str).str.strip()

print(f'\nSAM after cleaning: {len(sam_clean):,} rows')
sam_clean.head(3)

SAM before cleaning: 399,697 rows

SAM after cleaning: 399,697 rows


,SAM_ADDRESS_ID,BUILDING_ID,RELATIONSHIP_TYPE,FULL_ADDRESS,STREET_NUMBER,IS_RANGE,RANGE_FROM,RANGE_TO,UNIT,FULL_STREET_NAME,STREET_ID,STREET_PREFIX,STREET_BODY,STREET_SUFFIX_ABBR,STREET_FULL_SUFFIX,STREET_SUFFIX_DIR,STREET_NUMBER_SORT,MAILING_NEIGHBORHOOD,ZIP_CODE,X_COORD,Y_COORD,SAM_STREET_ID,WARD,PARCEL,created_date,last_edited_date,shape_wkt,POINT_X,POINT_Y
0,1,100778,1,6-10 A St,6-10,1,6,10,NaN,A St,1,NaN,A,St,Street,NaN,6,Hyde Park,2136,757684.428458,2.916575e+06,1,18,1809309000,9/25/2009 10:14:59,10/25/2017 14:04:04,POINT (-71.125035941999954 42.250617902000045),-71.125036,42.250618
1,11,154909,1,10 A St,10,0,NaN,NaN,NaN,A St,2,NaN,A,St,Street,NaN,10,South Boston,2127,775987.559175,2.949557e+06,2,6,0600090000,9/25/2009 10:14:59,2/10/2022 10:47:25,POINT (-71.056799999999953 42.340880001000073),-71.056800,42.340880
2,17,141252,1,176-178 A St,176-178,1,176,178,NaN,A St,2,NaN,A,St,Street,NaN,176,Boston,2210,776990.886893,2.951048e+06,2,6,0601169000,9/25/2009 10:14:59,2/10/2022 10:47:31,POINT (-71.053059999999959 42.344958000000076),-71.053060,42.344958


In [7]:
print(f'Property Assessment before cleaning: {len(pa):,} rows')

# Drop rows missing PID
pa_clean = pa.dropna(subset=['PID'])

# Drop dups
pa_clean = pa_clean.drop_duplicates(subset=['PID'])

# Strip whitespace
pa_clean['PID'] = pa_clean['PID'].str.strip()

if 'OWNER' in pa_clean.columns:
    pa_clean['OWNER'] = pa_clean['OWNER'].str.strip().str.upper()

# Cast YR_BUILT to int
if 'YR_BUILT' in pa_clean.columns:
    pa_clean['YR_BUILT'] = pd.to_numeric(pa_clean['YR_BUILT'], errors='coerce')

# Cap YR_BUILT
pa_clean = pa_clean[(pa_clean['YR_BUILT'].isna()) | 
                    ((pa_clean['YR_BUILT'] >= 1600) & (pa_clean['YR_BUILT'] <= 2025))]
print(f'\nProperty Assessment after cleaning: {len(pa_clean):,} rows')
pa_clean.head(3)

Property Assessment before cleaning: 184,552 rows

Property Assessment after cleaning: 184,551 rows


,PID,CM_ID,GIS_ID,ST_NUM,ST_NUM2,ST_NAME,UNIT_NUM,CITY,ZIP_CODE,BLDG_SEQ,NUM_BLDGS,LUC,LU,LU_DESC,BLDG_TYPE,OWN_OCC,OWNER,MAIL_ADDRESSEE,MAIL_STREET_ADDRESS,MAIL_CITY,MAIL_STATE,MAIL_ZIP_CODE,RES_FLOOR,CD_FLOOR,RES_UNITS,...,EXT_FNISHED,INT_COND,EXT_COND,OVERALL_COND,BED_RMS,FULL_BTH,HLF_BTH,KITCHENS,TT_RMS,BDRM_COND,BTHRM_STYLE1,BTHRM_STYLE2,BTHRM_STYLE3,KITCHEN_TYPE,KITCHEN_STYLE1,KITCHEN_STYLE2,KITCHEN_STYLE3,HEAT_TYPE,HEAT_SYSTEM,AC_TYPE,FIREPLACES,ORIENTATION,NUM_PARKING,PROP_VIEW,CORNER_UNIT
0,0100001000,NaN,100001000,195.0,NaN,Lexington ST,NaN,EAST BOSTON,2128.0,1,1,105,R3,THREE-FAM DWELLING,RE - Row End,Y,PASCUCCI CARLO,NaN,195 LEXINGTON ST,EAST BOSTON,MA,02128,3.0,NaN,NaN,...,A - Asbestos,A - Average,F - Fair,A - Average,6.0,3.0,0.0,3.0,12.0,NaN,S - Semi-Modern,S - Semi-Modern,S - Semi-Modern,3F - 3 Full Eat In Kitchens,S - Semi-Modern,S - Semi-Modern,S - Semi-Modern,W - Ht Water/Steam,NaN,N - None,0.0,NaN,3.0,A - Average,NaN
1,0100002000,NaN,100002000,197.0,NaN,Lexington ST,NaN,EAST BOSTON,2128.0,1,1,105,R3,THREE-FAM DWELLING,RM - Row Middle,N,SEMBRANO LIVING TRUST,NaN,2 MITCHELL LANE,WAKEFIELD,MA,01880,3.0,NaN,NaN,...,M - Vinyl,A - Average,A - Average,A - Average,3.0,3.0,0.0,3.0,9.0,NaN,M - Modern,M - Modern,M - Modern,3F - 3 Full Eat In Kitchens,M - Modern,M - Modern,M - Modern,F - Forced Hot Air,NaN,C - Central AC,0.0,NaN,0.0,A - Average,NaN
2,0100003000,NaN,100003000,199.0,NaN,Lexington ST,NaN,EAST BOSTON,2128.0,1,1,105,R3,THREE-FAM DWELLING,RM - Row Middle,N,GUERRA CHEVARRIA ANA S,NaN,199 LEXINGTON ST,EAST BOSTON,MA,02128,3.0,NaN,NaN,...,M - Vinyl,A - Average,G - Good,A - Average,5.0,3.0,0.0,3.0,13.0,NaN,M - Modern,M - Modern,M - Modern,3F - 3 Full Eat In Kitchens,S - Semi-Modern,S - Semi-Modern,S - Semi-Modern,S - Space Heat,NaN,N - None,0.0,NaN,0.0,A - Average,NaN


In [8]:
# Merge Combined Violations + SAM Addresses
merged_1 = violations.merge(
    sam_clean,
    left_on='sam_id',
    right_on='SAM_ADDRESS_ID',
    how='left',
    suffixes=('_bv', '_sam')
)

# Merge Result + Property Assessment
merged_2 = merged_1.merge(
    pa_clean,
    left_on='PARCEL',
    right_on='PID',
    how='left',
    suffixes=('', '_pa')
)

matched_2    = merged_2['PID'].notna().sum()
unmatched_2  = merged_2['PID'].isna().sum()
match_rate_2 = matched_2 / len(merged_2) * 100

print(f'\nAfter Merge:')
print(f'Total rows: {len(merged_2):,}')
print(f'Matched: {matched_2:,} ({match_rate_2:.1f}%)')
print(f'Unmatched: {unmatched_2:,} ({100 - match_rate_2:.1f}%)')


After Merge:
Total rows: 905,678
Matched: 860,103 (95.0%)
Unmatched: 45,575 (5.0%)


In [9]:
print('All columns in merged dataset:')
for col in merged_2.columns:
    print(' ', col)

All columns in merged dataset:
  case_no
  status_dttm
  status
  code
  value
  description
  violation_stno
  violation_sthigh
  violation_street
  violation_suffix
  violation_city
  violation_state
  violation_zip
  ward
  contact_addr1
  contact_addr2
  contact_city
  contact_state
  contact_zip
  sam_id
  latitude
  longitude
  source
  SAM_ADDRESS_ID
  BUILDING_ID
  RELATIONSHIP_TYPE
  FULL_ADDRESS
  STREET_NUMBER
  IS_RANGE
  RANGE_FROM
  RANGE_TO
  UNIT
  FULL_STREET_NAME
  STREET_ID
  STREET_PREFIX
  STREET_BODY
  STREET_SUFFIX_ABBR
  STREET_FULL_SUFFIX
  STREET_SUFFIX_DIR
  STREET_NUMBER_SORT
  MAILING_NEIGHBORHOOD
  ZIP_CODE
  X_COORD
  Y_COORD
  SAM_STREET_ID
  WARD
  PARCEL
  created_date
  last_edited_date
  shape_wkt
  POINT_X
  POINT_Y
  PID
  CM_ID
  GIS_ID
  ST_NUM
  ST_NUM2
  ST_NAME
  UNIT_NUM
  CITY
  ZIP_CODE_pa
  BLDG_SEQ
  NUM_BLDGS
  LUC
  LU
  LU_DESC
  BLDG_TYPE
  OWN_OCC
  OWNER
  MAIL_ADDRESSEE
  MAIL_STREET_ADDRESS
  MAIL_CITY
  MAIL_STATE
  MAIL_ZIP_CODE

In [10]:
keep_cols = [
    # From BV/PW — violation info
    'case_no',              # unique violation identifier
    'source',               # dataset origin: building_violations or public_works
    'status_dttm',          # date of violation
    'status',               # open/closed
    'code',                 # violation code
    'description',          # violation type
    'violation_stno',       # street number
    'violation_street',     # street name
    'violation_city',       # neighborhood
    'violation_zip',        # zip code
    'ward',                 # city ward
    'sam_id',               # join key
    'latitude',             # for mapping
    'longitude',

    # From SAM — location info
    'MAILING_NEIGHBORHOOD', # neighborhood
    'PARCEL',               # join key to PA
    'POINT_X',              # coordinates for mapping
    'POINT_Y',

    # From PA — property/owner info
    'OWNER',                # landlord/management company
    'YR_BUILT',             # build year
    'LU',                   # land use code
    'LU_DESC',              # land use description
    'BLDG_TYPE',            # building type
    'OWN_OCC',              # owner occupied Y/N
    'OVERALL_COND',         # building condition
]

df_final = merged_2[keep_cols]
print(f'Final dataset: {df_final.shape[0]:,} rows, {df_final.shape[1]} columns')
df_final.head(3)

Final dataset: 905,678 rows, 25 columns


,case_no,source,status_dttm,status,code,description,violation_stno,violation_street,violation_city,violation_zip,ward,sam_id,latitude,longitude,MAILING_NEIGHBORHOOD,PARCEL,POINT_X,POINT_Y,OWNER,YR_BUILT,LU,LU_DESC,BLDG_TYPE,OWN_OCC,OVERALL_COND
0,V91983,building_violations,NaT,Closed,121.2,Unsafe and Dangerous,302,Sumner,East Boston,02128,01,132380,42.367679,-71.036580,East Boston,0104910000,-71.036580,42.367679,ORELLANA-SERRANO ISRAEL,1930.0,R3,THREE-FAM DWELLING,RM - Row Middle,Y,A - Average
1,V903743,building_violations,2026-04-21 10:19:12,Open,102.8,Maintenance,125,B,South Boston,02127,06,7362,42.341270,-71.053080,South Boston,0601360000,-71.053080,42.341270,LAWRENCE PLACE CONDO TR,1899.0,CM,CONDO MAIN,LR - Low Rise,N,G - Good
2,V903738,building_violations,2026-04-21 10:07:25,Open,102.8,Maintenance,105,Third,South Boston,02127,06,135205,42.341123,-71.052757,South Boston,0601362000,-71.052757,42.341123,NOOK HILL CONDOMINIUM TRUST,2018.0,CM,CONDO MAIN,RM - Row Middle,N,A - Average


In [11]:
df_final.to_csv('merged_violations.csv', index=False)
print('Saved merged_violations_clean.csv')
print(f'Shape: {df_final.shape}')

Saved merged_violations_clean.csv
Shape: (905678, 25)
